# Analytical Fourier-optics validation

This notebook is the visual counterpart of `tests/test_analytical_optics.py`. It compares Fiatlux with four standard results: the Airy pattern, a $2\pi$ piston, the focal displacement generated by a phase ramp, and Parseval flux conservation.

In [ ]:
import matplotlib.pyplot as plt
import torch

from fiatlux.core.grid import Grid
from fiatlux.core.source import PlaneWave
from fiatlux.core.spectrum import Band, Spectrum
from fiatlux.optics.elements.mask import CircularAperture, Piston, TipTilt
from fiatlux.optics.propagator import MFTPropagator

torch.set_default_dtype(torch.float64)

In [ ]:
wavelength = 1e-6
focal_length = 2.0
diameter = 1.0
n_pupil = 128

pupil_grid = Grid(
    n_pupil, n_pupil, diameter / n_pupil, diameter / n_pupil,
    dtype=torch.float64,
)
spectrum = Spectrum(
    magnitude=0, band=Band(wavelength, 0.0, 1e8), samples=1,
    dtype=torch.float64,
)
plane_wave = PlaneWave(spectrum).generate_field(pupil_grid)
pupil_field = CircularAperture(pupil_grid, diameter / 2).apply(plane_wave)

## Circular aperture and first Airy minimum

For a circular aperture, the first intensity zero is expected at $1.21967\,\lambda/D$.

In [ ]:
pixels_per_lambda_d = 20
focal_sampling = wavelength * focal_length / diameter / pixels_per_lambda_d
focal_grid = Grid(128, 128, focal_sampling, focal_sampling, dtype=torch.float64)
focal_field = MFTPropagator(focal_length, focal_grid).apply(pupil_field)
psf = focal_field.intensity()[0]
psf_normalized = psf / psf.max()
center = focal_grid.nx // 2
radius_lambda_d = focal_grid.x[center:] / (wavelength * focal_length / diameter)
radial_cut = psf_normalized[center, center:]
search = (radius_lambda_d > 0.9) & (radius_lambda_d < 1.5)
candidate_indices = torch.nonzero(search).flatten()
zero_index = candidate_indices[radial_cut[search].argmin()]
measured_zero = radius_lambda_d[zero_index].item()
print(f'Measured first minimum: {measured_zero:.3f} lambda/D')
print('Theory: 1.21967 lambda/D')

In [ ]:
extent = [value.item() for value in (
    focal_grid.x.min() / (wavelength * focal_length / diameter),
    focal_grid.x.max() / (wavelength * focal_length / diameter),
    focal_grid.y.min() / (wavelength * focal_length / diameter),
    focal_grid.y.max() / (wavelength * focal_length / diameter),
)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
image = axes[0].imshow(
    torch.log10(psf_normalized.clamp_min(1e-8)).numpy(),
    origin='lower', extent=extent, vmin=-6, vmax=0, cmap='magma',
)
axes[0].set(xlabel=r'$x/(\lambda f/D)$', ylabel=r'$y/(\lambda f/D)$', title='Airy PSF (log10)')
fig.colorbar(image, ax=axes[0], label='log10 normalized intensity')
axes[1].semilogy(radius_lambda_d, radial_cut, label='Fiatlux')
axes[1].axvline(1.21967, color='tab:red', linestyle='--', label='theory: 1.21967')
axes[1].scatter([measured_zero], [radial_cut[zero_index]], color='black', zorder=3, label=f'sampled: {measured_zero:.2f}')
axes[1].set(xlim=(0, 3), ylim=(1e-6, 1), xlabel=r'$r/(\lambda f/D)$', ylabel='Normalized intensity', title='Radial cut')
axes[1].grid(True, which='both', alpha=0.3)
axes[1].legend()
fig.tight_layout()

## One-wavelength piston

An OPD of one wavelength adds exactly $2\pi$ phase, so the complex field must be unchanged.

In [ ]:
piston_field = Piston(pupil_grid, piston=wavelength).apply(plane_wave)
phase_shift = 2 * torch.pi * wavelength / spectrum.wavelengths[0]
piston_error = (piston_field.complex_amplitude - plane_wave.complex_amplitude).abs().max()
print(f'Phase shift: {phase_shift.item():.15f} rad')
print(f'2 pi:        {(2 * torch.pi):.15f} rad')
print(f'Max complex-field error: {piston_error.item():.3e}')

## Tip/tilt displacement

A pupil ramp containing one phase cycle across $D$ moves the PSF by one Fourier bin.

In [ ]:
fourier_sampling = wavelength * focal_length / diameter
fourier_grid = Grid(64, 64, fourier_sampling, fourier_sampling, dtype=torch.float64)
small_grid = Grid(64, 64, diameter / 64, diameter / 64, dtype=torch.float64)
small_plane = PlaneWave(spectrum).generate_field(small_grid)
small_pupil = CircularAperture(small_grid, diameter / 2).apply(small_plane)
tilted_pupil = TipTilt(small_grid, tip=wavelength / diameter, tilt=0.0).apply(small_pupil)
propagator = MFTPropagator(focal_length, fourier_grid)
reference_psf = propagator.apply(small_pupil).intensity()[0]
tilted_psf = propagator.apply(tilted_pupil).intensity()[0]
reference_peak = torch.unravel_index(reference_psf.argmax(), reference_psf.shape)
tilted_peak = torch.unravel_index(tilted_psf.argmax(), tilted_psf.shape)
print('Reference peak (y, x):', tuple(index.item() for index in reference_peak))
print('Tilted peak   (y, x):', tuple(index.item() for index in tilted_peak))
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, image, title in zip(axes, [reference_psf, tilted_psf], ['Reference', 'One-cycle tip']):
    ax.imshow((image / image.max()).numpy(), origin='lower', cmap='magma')
    ax.set_title(title)
    ax.set_xlim(27, 38)
    ax.set_ylim(27, 38)
fig.tight_layout()

## Flux conservation

On conjugate pupil and focal grids, the lossless MFT must preserve the spatially integrated intensity.

In [ ]:
input_flux = small_pupil.intensity().sum() * small_grid.dx * small_grid.dy
output_flux = tilted_psf.sum() * fourier_grid.dx * fourier_grid.dy
relative_error = ((output_flux - input_flux) / input_flux).abs()
print(f'Input flux:          {input_flux.item():.12e} photons/s')
print(f'Output flux:         {output_flux.item():.12e} photons/s')
print(f'Relative difference: {relative_error.item():.3e}')
assert relative_error < 1e-12